# 🔧 Day 33: Machine Learning Part 2 — Feature Engineering

---

## 📋 Aaj Ke Topics:

| # | Topic |
|---|-------|
| 1 | Feature Engineering kya hoti hai? |
| 2 | Missing Values Handle karna |
| 3 | Categorical Variables Encode karna |
| 4 | Feature Scaling (StandardScaler, MinMaxScaler, RobustScaler) |
| 5 | Feature Transformation (Log, Sqrt, Box-Cox) |
| 6 | Naye Features Banana (Interaction Terms, Polynomial) |
| 7 | Feature Selection (Correlation, VarianceThreshold, SelectKBest) |
| 8 | Outliers Handle karna (IQR + Z-score) |

---
**Dataset:** `insurance.csv` — Medical charges dataset (1338 rows, 7 features)

## 1️⃣ Feature Engineering Kya Hoti Hai?

**Definition:**  
> Raw data ko ML model ke liye better, meaningful aur useful format mein transform karna — **Feature Engineering** kehlata hai.

### Kyun Zaroori Hai?
- ML model sirf numbers samajhta hai — text, categories, missing values seedha use nahi kar sakta
- **Garbage In → Garbage Out** — agar data bekar hai toh model bhi bekar hoga
- Achhi Feature Engineering se model ki accuracy dramatically improve hoti hai

### Feature Engineering ke Steps:
```
Raw Data
   ↓
Missing Values Handle
   ↓
Categorical Encoding
   ↓
Feature Scaling
   ↓
Feature Transformation
   ↓
New Feature Creation
   ↓
Feature Selection
   ↓
Model Training ✅
```

In [ ]:
# --- Libraries Import ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Dataset load karo
df = pd.read_csv('insurance.csv')
print("Dataset Shape:", df.shape)
print("\nPehli 5 Rows:")
df.head()

## 2️⃣ Missing Values Handle Karna

**Missing values** woh cells hoti hain jahan data nahi hota (NaN / null).  
ML models missing values ke saath kaam nahi kar sakte — isliye inhe handle karna zaroori hai.

### 2 Main Strategies:

| Strategy | Kab Use Karein |
|----------|----------------|
| **Drop** (`.dropna()`) | Jab missing rows bahut kam hon (< 5%) |
| **Impute** (fill karo) | Jab missing data zyada ho — drop karna nuksaan |

### Imputation Methods (`SimpleImputer`):
- `strategy='mean'` → Numeric columns ke liye (normal distribution)
- `strategy='median'` → Numeric columns (skewed data / outliers hoon)
- `strategy='most_frequent'` → Categorical columns ke liye
- `strategy='constant'` → Apni value dena ho (e.g., 0 ya 'Unknown')

In [ ]:
from sklearn.impute import SimpleImputer

# Insurance data mein missing values check karo
print("=== Missing Values Check ===")
print(df.isnull().sum())
print(f"\nTotal Missing: {df.isnull().sum().sum()}")

# --- Demo: Artificially missing values banana aur phir fix karna ---
df_demo = df.copy()

# 10% rows mein age aur bmi ko NaN kar do (demo ke liye)
np.random.seed(42)
missing_idx = np.random.choice(df_demo.index, size=130, replace=False)
df_demo.loc[missing_idx[:65], 'age'] = np.nan
df_demo.loc[missing_idx[65:], 'bmi'] = np.nan

print("\n--- Demo Missing Values (Artificially Created) ---")
print(df_demo.isnull().sum())

# --- Mean Imputation ---
imputer_mean = SimpleImputer(strategy='mean')
df_demo[['age', 'bmi']] = imputer_mean.fit_transform(df_demo[['age', 'bmi']])

print("\n--- After Mean Imputation ---")
print(df_demo.isnull().sum())

# --- Visualization: Before vs After ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Missing Values: Before vs After Imputation', fontsize=14, fontweight='bold')

# Original age distribution
axes[0].hist(df['age'], bins=20, color='steelblue', alpha=0.7, label='Original')
axes[0].set_title('Age Distribution (Original)')
axes[0].set_xlabel('Age')
axes[0].legend()

# After imputation age distribution
axes[1].hist(df_demo['age'], bins=20, color='tomato', alpha=0.7, label='After Imputation')
axes[1].set_title('Age Distribution (After Mean Imputation)')
axes[1].set_xlabel('Age')
axes[1].legend()

plt.tight_layout()
plt.show()
print("✅ Mean imputation karne ke baad distribution roughly same rehti hai!")

## 3️⃣ Categorical Variables Encode Karna

ML models sirf **numbers** samajhte hain — 'male', 'female', 'yes', 'no' jaise strings seedha use nahi ho sakti.  
Isliye hum inhe numbers mein convert karte hain — ise **Encoding** kehte hain.

### 2 Main Types:

| Encoding | Kab Use | Example |
|----------|---------|--------|
| **Label Encoding** | Ordinal (order matter kare) | Small=0, Medium=1, Large=2 |
| **One-Hot Encoding** | Nominal (no order) | red=[1,0,0], blue=[0,1,0] |

### ⚠️ Label Encoding ka Problem:
Agar category mein koi order nahi (like: regions — northeast, southwest...) toh Label Encoding galat assumption deta hai ke `northeast > southwest` — jo bilkul galat hai!  
→ Aise cases mein **One-Hot Encoding** use karo.

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_enc = df.copy()

# --- Label Encoding: smoker (yes/no → 1/0) ---
le = LabelEncoder()
df_enc['smoker_encoded'] = le.fit_transform(df_enc['smoker'])  # no=0, yes=1
df_enc['sex_encoded'] = le.fit_transform(df_enc['sex'])        # female=0, male=1

print("=== Label Encoding ===")
print(df_enc[['smoker', 'smoker_encoded', 'sex', 'sex_encoded']].head(8))

# --- One-Hot Encoding: region (4 categories) ---
df_ohe = pd.get_dummies(df_enc, columns=['region'], prefix='region', drop_first=False)

print("\n=== One-Hot Encoding: Region ===")
region_cols = [c for c in df_ohe.columns if 'region_' in c]
print(df_ohe[['region_northeast', 'region_northwest', 'region_southeast', 'region_southwest']].head(8))

# --- Visualization: Category Counts ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Categorical Columns Distribution', fontsize=13, fontweight='bold')

df['smoker'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','tomato'], edgecolor='black')
axes[0].set_title('Smoker'); axes[0].set_xlabel(''); axes[0].tick_params(rotation=0)

df['sex'].value_counts().plot(kind='bar', ax=axes[1], color=['orchid','skyblue'], edgecolor='black')
axes[1].set_title('Sex'); axes[1].set_xlabel(''); axes[1].tick_params(rotation=0)

df['region'].value_counts().plot(kind='bar', ax=axes[2], color='steelblue', edgecolor='black')
axes[2].set_title('Region'); axes[2].set_xlabel(''); axes[2].tick_params(rotation=30)

plt.tight_layout()
plt.show()
print(f"\n✅ Shape after One-Hot Encoding: {df_ohe.shape} (pehle: {df.shape})")

## 4️⃣ Feature Scaling ⭐ (Bohot Important!)

Humara data mein alag alag features ki **scale** bilkul mukhtalif hoti hai:  
- `age` → 18 to 64  
- `bmi` → 15 to 55  
- `charges` → 1,000 to 65,000  

Kuch algorithms (Linear Regression, KNN, SVM, Neural Networks) is scale difference se affect hote hain — bada number model ko dominate karta hai.  
**Scaling** se sab features ek range mein aa jaate hain.

### 3 Main Scalers:

| Scaler | Formula | Range | Kab Use |
|--------|---------|-------|--------|
| **StandardScaler** | `(x − μ) / σ` | mean=0, std=1 | Normal distribution, no extreme outliers |
| **MinMaxScaler** | `(x − min) / (max − min)` | 0 to 1 | Jab range fix chahiye |
| **RobustScaler** | `(x − median) / IQR` | Varies | Outliers bahut zyada hoon |

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Age column use karte hain demo ke liye
age_data = df[['age']].values

std_scaler   = StandardScaler()
minmax_scaler = MinMaxScaler()
robust_scaler = RobustScaler()

age_std    = std_scaler.fit_transform(age_data)
age_minmax = minmax_scaler.fit_transform(age_data)
age_robust = robust_scaler.fit_transform(age_data)

# --- Stats comparison ---
print("=" * 55)
print(f"{'Feature':<18} {'Min':>8} {'Max':>8} {'Mean':>8} {'Std':>8}")
print("=" * 55)
print(f"{'Original Age':<18} {age_data.min():>8.2f} {age_data.max():>8.2f} {age_data.mean():>8.2f} {age_data.std():>8.2f}")
print(f"{'StandardScaler':<18} {age_std.min():>8.2f} {age_std.max():>8.2f} {age_std.mean():>8.2f} {age_std.std():>8.2f}")
print(f"{'MinMaxScaler':<18} {age_minmax.min():>8.2f} {age_minmax.max():>8.2f} {age_minmax.mean():>8.2f} {age_minmax.std():>8.2f}")
print(f"{'RobustScaler':<18} {age_robust.min():>8.2f} {age_robust.max():>8.2f} {age_robust.mean():>8.2f} {age_robust.std():>8.2f}")
print("=" * 55)

# --- Visualization: 3 Scalers side by side ---
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Feature Scaling Comparison — Age Column', fontsize=13, fontweight='bold')

data_list   = [age_data, age_std, age_minmax, age_robust]
titles      = ['Original\n(age)', 'StandardScaler\n(mean=0, std=1)', 'MinMaxScaler\n(0 to 1)', 'RobustScaler\n(IQR based)']
colors      = ['steelblue', 'tomato', 'mediumseagreen', 'darkorange']

for ax, data, title, color in zip(axes, data_list, titles, colors):
    ax.hist(data, bins=20, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Value')

plt.tight_layout()
plt.show()
print("✅ Shape same rehti hai — sirf values ki scale badal jaati hai!")

## 5️⃣ Feature Transformation

Kuch features ki distribution **skewed** hoti hai — ek taraf zyada data, doosri taraf kam.  
Linear Regression jaise models normal distribution assume karte hain — skewed data se performance kharab hoti hai.

### Transformations:

| Transform | Formula | Kab Use |
|-----------|---------|--------|
| **Log** | `log(x)` ya `log(x+1)` | Right-skewed data (income, charges) |
| **Square Root** | `√x` | Mild skew, count data |
| **Box-Cox** | Auto-optimal power transform | Jab exact transform pata na ho |

**Humara `charges` column right-skewed hai** — Day 32 mein bhi dekha tha!  
Log transform se isko normal-like banana possible hai.

In [ ]:
from scipy import stats

charges = df['charges']
log_charges  = np.log(charges)
sqrt_charges = np.sqrt(charges)
boxcox_charges, lam = stats.boxcox(charges)

# Skewness values
print("=== Skewness Comparison ===")
print(f"Original charges Skewness : {charges.skew():.4f}")
print(f"Log Transform  Skewness   : {log_charges.skew():.4f}")
print(f"Sqrt Transform Skewness   : {sqrt_charges.skew():.4f}")
print(f"Box-Cox (λ={lam:.3f}) Skewness: {pd.Series(boxcox_charges).skew():.4f}")
print("\n👉 0 ke qareeb = zyada normal distribution!")

# --- Visualization ---
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Feature Transformation — charges Column', fontsize=13, fontweight='bold')

datasets = [charges, log_charges, sqrt_charges, boxcox_charges]
titles   = [
    f'Original (Skew={charges.skew():.2f})',
    f'Log Transform (Skew={log_charges.skew():.2f})',
    f'Sqrt Transform (Skew={sqrt_charges.skew():.2f})',
    f'Box-Cox λ={lam:.2f} (Skew={pd.Series(boxcox_charges).skew():.2f})'
]
clrs = ['tomato', 'steelblue', 'mediumseagreen', 'darkorchid']

for ax, data, title, c in zip(axes.flat, datasets, titles, clrs):
    ax.hist(data, bins=35, color=c, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Value')

plt.tight_layout()
plt.show()
print("✅ Log / Box-Cox transform ke baad charges ka distribution normal ho gaya!")

## 6️⃣ Naye Features Banana (Feature Creation)

Kabhi kabhi existing features se **naye features** banana model ko better predictions deta hai.

### Methods:

1. **Interaction Terms** — Do features ko multiply karo  
   → Jaise Day 32 mein: `bmi_smoker = bmi × smoker` — iska coefficient `256.9` tha!  
   → Matlab: ek smoker ka charge per unit BMI aur zyada badhta hai

2. **Polynomial Features** — `x²`, `x³`, interactions automatically banana  
   → Non-linear relationships capture karne ke liye

3. **Domain Knowledge Features**  
   → Jaise: `bmi_category` (Underweight / Normal / Overweight / Obese)  
   → `age_group` (Young / Middle / Senior)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, LabelEncoder

df_feat = df.copy()

# --- 1. Interaction Term: bmi × smoker (Day 32 se) ---
le = LabelEncoder()
df_feat['smoker_num'] = le.fit_transform(df_feat['smoker'])
df_feat['bmi_smoker'] = df_feat['bmi'] * df_feat['smoker_num']
print("=== Interaction Feature: bmi_smoker ===")
print(df_feat[['bmi', 'smoker', 'smoker_num', 'bmi_smoker']].head(8))

# --- 2. Domain Knowledge: BMI Category ---
def bmi_category(bmi):
    if bmi < 18.5: return 'Underweight'
    elif bmi < 25:  return 'Normal'
    elif bmi < 30:  return 'Overweight'
    else:           return 'Obese'

df_feat['bmi_cat'] = df_feat['bmi'].apply(bmi_category)
print("\n=== BMI Category Feature ===")
print(df_feat['bmi_cat'].value_counts())

# --- 3. Polynomial Features (age, bmi) ---
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(df_feat[['age', 'bmi']])
poly_names = poly.get_feature_names_out(['age', 'bmi'])
print("\n=== Polynomial Features (degree=2) ===")
print("Features generated:", list(poly_names))
print("Shape:", X_poly.shape)

# --- Visualization: Charges by BMI Category ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('New Features Analysis', fontsize=13, fontweight='bold')

# Boxplot: charges by BMI category
order = ['Underweight', 'Normal', 'Overweight', 'Obese']
sns.boxplot(data=df_feat, x='bmi_cat', y='charges', order=order,
            palette='Set2', ax=axes[0])
axes[0].set_title('Charges by BMI Category')
axes[0].set_xlabel('BMI Category')

# Scatter: bmi_smoker vs charges
colors = df_feat['smoker_num'].map({0: 'steelblue', 1: 'tomato'})
axes[1].scatter(df_feat['bmi_smoker'], df_feat['charges'], c=colors, alpha=0.5, s=20)
axes[1].set_title('bmi_smoker vs Charges')
axes[1].set_xlabel('bmi × smoker')
axes[1].set_ylabel('Charges')
from matplotlib.patches import Patch
legend_elem = [Patch(color='steelblue', label='Non-Smoker'), Patch(color='tomato', label='Smoker')]
axes[1].legend(handles=legend_elem)

plt.tight_layout()
plt.show()
print("✅ bmi_smoker feature smokers ke charges alag clearly dikhata hai!")

## 7️⃣ Feature Selection

Jitne features zyada, utna model complex — but **sab features useful nahi hote**.  
Kuch features:
- Dusre features se bohot **correlated** hain (redundant)
- **Near-zero variance** hain (koi information nahi)
- Target ke saath koi **statistical relation** nahi

**Feature Selection** se hum sirf **useful features** rakhte hain.

### Methods:
| Method | Kya Karta Hai |
|--------|---------------|
| **Correlation Filter** | High correlated features drop karo (> 0.9) |
| **VarianceThreshold** | Near-zero variance features drop karo |
| **SelectKBest** | Statistical test se top K features select karo |
| **Model Coefficients** | Trained model ke weights dekho (kaunsa feature important) |

In [ ]:
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression

# Full numeric dataset banana
df_sel = df.copy()
le = LabelEncoder()
df_sel['sex']    = le.fit_transform(df_sel['sex'])
df_sel['smoker'] = le.fit_transform(df_sel['smoker'])
df_sel['region'] = le.fit_transform(df_sel['region'])
df_sel['bmi_smoker'] = df_sel['bmi'] * df_sel['smoker']

X = df_sel.drop('charges', axis=1)
y = df_sel['charges']

# --- 1. Correlation Heatmap ---
print("=== 1. Correlation Matrix ===")
corr = df_sel.corr()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Feature Selection Analysis', fontsize=13, fontweight='bold')

sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[0], linewidths=0.5)
axes[0].set_title('Correlation Heatmap')

# --- 2. SelectKBest: Top 5 Features ---
selector = SelectKBest(score_func=f_regression, k=5)
selector.fit(X, y)
feature_scores = pd.DataFrame({
    'Feature': X.columns,
    'Score': selector.scores_
}).sort_values('Score', ascending=True)

colors_bar = ['tomato' if s == feature_scores['Score'].max() else 'steelblue'
              for s in feature_scores['Score']]
axes[1].barh(feature_scores['Feature'], feature_scores['Score'],
             color=colors_bar, edgecolor='white')
axes[1].set_title('SelectKBest — Feature Importance Scores\n(f_regression)')
axes[1].set_xlabel('F-Score')

plt.tight_layout()
plt.show()

# --- 3. VarianceThreshold ---
vt = VarianceThreshold(threshold=0.0)
vt.fit(X)
print("\n=== VarianceThreshold ===")
print("Features with zero variance (drop):",
      list(X.columns[~vt.get_support()]))
print("Features kept:", list(X.columns[vt.get_support()]))

# --- 4. Top features from SelectKBest ---
top5 = feature_scores.sort_values('Score', ascending=False).head(5)
print("\n=== Top 5 Features (SelectKBest) ===")
print(top5.to_string(index=False))
print("\n✅ bmi_smoker sabse zyada important feature hai — Day 32 ki baat sahi thi!")

## 8️⃣ Outliers Handle Karna

**Outliers** woh data points hain jo baaki data se bohot door hote hain.  
Ye model ko galat direction mein train kar sakte hain — especially Linear Regression.

### 2 Popular Methods:

**Method 1: IQR (Interquartile Range)**
```
Q1 = 25th percentile
Q3 = 75th percentile
IQR = Q3 - Q1

Lower Fence = Q1 - 1.5 × IQR
Upper Fence = Q3 + 1.5 × IQR

Outlier = value < Lower OR value > Upper
```

**Method 2: Z-Score**
```
Z = (x - mean) / std

Outlier = |Z| > 3
```

→ **IQR method zyada robust hai** (outliers se affect nahi hoti mean/std ki tarah)

In [ ]:
# --- IQR Method ---
charges = df['charges']

Q1 = charges.quantile(0.25)
Q3 = charges.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_iqr = df[(charges < lower) | (charges > upper)]
df_no_outliers_iqr = df[(charges >= lower) & (charges <= upper)]

print("=== IQR Method ===")
print(f"Q1={Q1:.2f}  Q3={Q3:.2f}  IQR={IQR:.2f}")
print(f"Lower Fence = {lower:.2f}")
print(f"Upper Fence = {upper:.2f}")
print(f"Outliers found (IQR): {len(outliers_iqr)}")
print(f"Data after removal: {len(df_no_outliers_iqr)} rows (pehle: {len(df)})")

# --- Z-Score Method ---
from scipy import stats
z_scores = np.abs(stats.zscore(charges))
outliers_z = df[z_scores > 3]
df_no_outliers_z = df[z_scores <= 3]

print("\n=== Z-Score Method (|Z| > 3) ===")
print(f"Outliers found (Z-score): {len(outliers_z)}")
print(f"Data after removal: {len(df_no_outliers_z)} rows")

# --- Visualization: Boxplot Before vs After ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Outlier Detection & Removal — charges Column', fontsize=13, fontweight='bold')

axes[0].boxplot(charges, patch_artist=True,
                boxprops=dict(facecolor='tomato', alpha=0.6))
axes[0].set_title(f'Original\n({len(df)} rows)')
axes[0].set_ylabel('Charges')

axes[1].boxplot(df_no_outliers_iqr['charges'], patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title(f'After IQR Removal\n({len(df_no_outliers_iqr)} rows)')

axes[2].boxplot(df_no_outliers_z['charges'], patch_artist=True,
                boxprops=dict(facecolor='mediumseagreen', alpha=0.6))
axes[2].set_title(f'After Z-Score Removal\n({len(df_no_outliers_z)} rows)')

plt.tight_layout()
plt.show()
print("✅ Outlier removal ke baad distribution zyada compact ho jati hai!")

## 🎯 Day 33 Summary — Feature Engineering Pipeline

```
Raw Data (insurance.csv)
    │
    ├── 1. Missing Values  → SimpleImputer (mean/median/mode)
    │
    ├── 2. Encoding        → LabelEncoder (binary), OneHotEncoder (multi-category)
    │
    ├── 3. Scaling         → StandardScaler / MinMaxScaler / RobustScaler
    │
    ├── 4. Transformation  → Log / Sqrt / Box-Cox (skew fix karo)
    │
    ├── 5. New Features    → bmi×smoker, BMI Category, PolynomialFeatures
    │
    ├── 6. Feature Select  → Correlation, VarianceThreshold, SelectKBest
    │
    └── 7. Outliers        → IQR Method / Z-Score Method
            │
            ▼
    Clean, Engineered Data → Model Training ✅
```

### 🔑 Key Takeaways:
| Point | Lesson |
|-------|--------|
| `bmi_smoker` feature | Sabse powerful feature — R² 78% → 86% gaya |
| Scaling | Z-score / MinMax — same distribution, alag scale |
| Log Transform | Right-skewed `charges` column ko normal banaya |
| IQR > Z-score | Outlier detection mein IQR zyada robust hai |
| Garbage In = Garbage Out | Feature Engineering = Model ka foundation |

---
**Next: Day 34 — Regularization (Ridge & Lasso Regression)** 🚀